In [1]:
!pip install -q diffusers transformers accelerate ftfy hf_transfer

In [2]:
import torch
import torch.nn as nn
from diffusers import AutoencoderKLWan, WanPipeline

from taehv import TAEHV

class DotDict(dict):
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__

class TAEW2_1DiffusersWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        self.dtype = torch.float16
        self.device = "cuda"
        self.taehv = TAEHV("taew2_1.pth").to(self.dtype)
        self.temperal_downsample = [True, True, False] # [sic]
        self.config = DotDict(scaling_factor=1.0, latents_mean=torch.zeros(16), z_dim=16, latents_std=torch.ones(16))

    def decode(self, latents, return_dict=None):
        n, c, t, h, w = latents.shape
        # low-memory, set parallel=True for faster + higher memory
        return (self.taehv.decode_video(latents.transpose(1, 2), parallel=False).transpose(1, 2).mul_(2).sub_(1),)

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.vae = TAEW2_1DiffusersWrapper().to("cuda")
pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

WanPipeline {
  "_class_name": "WanPipeline",
  "_diffusers_version": "0.35.2",
  "_name_or_path": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
  "boundary_ratio": null,
  "expand_timesteps": false,
  "scheduler": [
    "diffusers",
    "UniPCMultistepScheduler"
  ],
  "text_encoder": [
    "transformers",
    "UMT5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "WanTransformer3DModel"
  ],
  "transformer_2": [
    null,
    null
  ],
  "vae": [
    "__main__",
    "TAEW2_1DiffusersWrapper"
  ]
}

In [7]:
output = pipe(
    prompt="a slice of delicious new-york style cheesecake with mint and berries is placed onto a plate and then drizzled with berry sauce",
    negative_prompt="Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards",
    height=480,
    width=832,
    num_frames=81,
    guidance_scale=5.0,
    generator=torch.Generator("cpu").manual_seed(sum(ord(c) for c in "TAEW2.1")),
    output_type="pil"
).frames[0]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

In [8]:
from IPython.display import HTML
from diffusers.utils import export_to_video

def frames_to_html(frames):
    """Display TCHW [0, 1] frame tensor as inline video."""
    export_to_video(frames, "decoded_taew2_1.mp4", fps=30)
    return HTML(f"<video playsinline autoplay loop muted><source src='decoded_taew2_1.mp4' type='video/mp4'></video>")

display(frames_to_html(output))

It is recommended to use `export_to_video` with `imageio` and `imageio-ffmpeg` as a backend. 
These libraries are not present in your environment. Attempting to use legacy OpenCV backend to export video. 
Support for the OpenCV backend will be deprecated in a future Diffusers version
